In [1]:
from langgraph.graph import StateGraph,END,add_messages
from typing import TypedDict,Annotated
from langgraph.types import Command,interrupt
from langgraph.checkpoint.memory import MemorySaver


/Users/mohan/Desktop/codebasics/Langraph/venv/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [27]:
memory = MemorySaver()

class graph_state(TypedDict):
    text :str

def node_a(state:graph_state):
    print("Node A")
    return Command(
        goto="node_b",
        update={
            "text" : state["text"] + "a"
        }
    )

def node_b(state:graph_state):
    print("Node B")
    human_input = interrupt("do you want to go to C or D, type C/D")

    if(human_input == "C"):
        return Command(
            goto="node_c",
            update={
                "text" : state["text"] + "b"
            }
        )
    elif(human_input == "D"):
        return Command(
            goto = "node_d",
            update={
                "text" : state["text"] + "b"
            }
        )
    
def node_c(state:graph_state):
    print("Node C")
    return Command(
        goto=END,
        update={
            "text" : state["text"] + "c"
        }
    ) 

def node_d(state:graph_state):
    print("Node D")
    return Command(
        goto=END,
        update={
            "text" : state["text"] + "d"
        }
    )

graph = StateGraph(graph_state)

graph.add_node("node_a",node_a)
graph.add_node("node_b",node_b)
graph.add_node("node_c",node_c)
graph.add_node("node_d",node_d)

graph.set_entry_point("node_a")

config = {
    "configurable" :{
        "thread_id" :1
    }
}

app = graph.compile(checkpointer=memory)

current_state = {
    "text" : ""
}
final_result = app.invoke(config=config,input=current_state,stream_mode="updates")
final_result

Node A
Node B


[{'node_a': {'text': 'a'}},
 {'__interrupt__': (Interrupt(value='do you want to go to C or D, type C/D', resumable=True, ns=['node_b:11a2c3e1-e6d3-428c-d9a5-b8b7f899918e'], when='during'),)}]

In [28]:
print(app.get_state(config).next)

('node_b',)


In [29]:
second_result = app.invoke(Command(resume="D"),config=config,stream_mode="updates")
print(second_result)

Node B
Node D
[{'node_b': {'text': 'ab'}}, {'node_d': {'text': 'abd'}}]
